# CoCoNut C 数据集去重

## 背景

经验证，`dataset/c/raw/` 下的 4 个文件各含 **2,735,506 行**，
其中第 1 ~ 1,367,753 行与第 1,367,754 ~ 2,735,506 行**完全重复**。

本 Notebook 以流式逐行方式保留前半部分，输出至 `dataset/c/dedup/`，**不修改原始文件**。

## Cell 1 — 配置

In [1]:
from pathlib import Path
import time

# ── 路径配置 ──────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"<DATASET_ROOT>\CoCoNut")
RAW_DIR   = BASE_DIR / "dataset" / "c" / "raw"
DEDUP_DIR = BASE_DIR / "dataset" / "c" / "dedup"

# ── 行数常量 ──────────────────────────────────────────────────────────────────
TOTAL_LINES = 2_735_506   # 每个文件的总行数
KEEP_LINES  = 1_367_753   # 保留前半部分（后半与前半完全重复）

# ── 待处理文件（context.txt 最大，排最后）────────────────────────────────────
FILES = ["meta.txt", "add.txt", "rem.txt", "context.txt"]

DEDUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"原始目录  : {RAW_DIR}")
print(f"输出目录  : {DEDUP_DIR}")
print(f"总行数    : {TOTAL_LINES:,}")
print(f"保留行数  : {KEEP_LINES:,}")
print(f"待处理文件: {FILES}")

原始目录  : <DATASET_ROOT>\CoCoNut\dataset\c\raw
输出目录  : <DATASET_ROOT>\CoCoNut\dataset\c\dedup
总行数    : 2,735,506
保留行数  : 1,367,753
待处理文件: ['meta.txt', 'add.txt', 'rem.txt', 'context.txt']


## Cell 2 — 预验证（抽样检查重复性）

从 `meta.txt`（最小、最易读）中抽取几个样本行，验证第 `N` 行与第 `N + KEEP_LINES` 行内容完全一致。

In [2]:
# 抽样行号（1-indexed），覆盖首行、中部、末尾
SAMPLE_ROWS = [1, 155, 1_000, 100_000, KEEP_LINES]

def sample_verify(file_path: Path, sample_rows: list, keep: int) -> bool:
    """逐行扫描文件，在目标行号处抽取内容，然后比较前后两半。"""
    targets = set(sample_rows) | {r + keep for r in sample_rows}
    captured = {}  # {行号: 内容}

    with open(file_path, 'r', encoding='utf-8') as f:
        for lineno, line in enumerate(f, start=1):
            if lineno in targets:
                captured[lineno] = line
            if lineno > max(targets):
                break

    all_match = True
    print(f"\n验证文件: {file_path.name}")
    print("-" * 60)
    for n in sample_rows:
        first  = captured.get(n, None)
        second = captured.get(n + keep, None)
        match  = (first == second)
        if not match:
            all_match = False
        status = "✓ 匹配" if match else "✗ 不匹配！"
        print(f"  行 {n:>9,} vs 行 {n + keep:>10,}  →  {status}")
        if not match:
            print(f"    前半: {repr(first[:80] if first else None)}")
            print(f"    后半: {repr(second[:80] if second else None)}")
    print("-" * 60)
    return all_match

meta_path = RAW_DIR / "meta.txt"
ok = sample_verify(meta_path, SAMPLE_ROWS, KEEP_LINES)
if ok:
    print("\n[预验证通过] 前半与后半数据完全重复，可以安全去重。")
else:
    print("\n[警告] 存在不匹配行，请检查 KEEP_LINES 配置后再继续！")


验证文件: meta.txt
------------------------------------------------------------
  行         1 vs 行  1,367,754  →  ✓ 匹配
  行       155 vs 行  1,367,908  →  ✓ 匹配
  行     1,000 vs 行  1,368,753  →  ✓ 匹配
  行   100,000 vs 行  1,467,753  →  ✓ 匹配
  行 1,367,753 vs 行  2,735,506  →  ✓ 匹配
------------------------------------------------------------

[预验证通过] 前半与后半数据完全重复，可以安全去重。


## Cell 3 — 去重核心函数

In [3]:
try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("未安装 tqdm，将使用简单进度输出。可运行 `pip install tqdm` 安装。")


def dedup_file(src: Path, dst: Path, keep: int, buf_size: int = 1 << 20) -> dict:
    """
    流式去重：仅保留 src 文件前 keep 行，写入 dst。

    参数:
        src      : 源文件路径
        dst      : 目标文件路径
        keep     : 保留的行数（1-indexed，含）
        buf_size : 写缓冲区大小（字节），默认 1 MB

    返回:
        {'lines_written': int, 'elapsed_s': float}
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    lines_written = 0

    with open(src, 'r', encoding='utf-8') as fin, \
         open(dst, 'w', encoding='utf-8', buffering=buf_size) as fout:

        if HAS_TQDM:
            pbar = tqdm(total=keep, desc=src.name, unit='行', unit_scale=True)

        for lineno, line in enumerate(fin, start=1):
            if lineno > keep:
                break
            fout.write(line)
            lines_written += 1

            if HAS_TQDM:
                pbar.update(1)
            elif lineno % 100_000 == 0:
                pct = lineno / keep * 100
                elapsed = time.time() - t0
                print(f"  {src.name}: {lineno:>10,} / {keep:,} 行 ({pct:.1f}%)  "
                      f"耗时 {elapsed:.1f}s", flush=True)

        if HAS_TQDM:
            pbar.close()

    elapsed = time.time() - t0
    return {'lines_written': lines_written, 'elapsed_s': elapsed}


print("去重函数已定义。")

去重函数已定义。


## Cell 4 — 批量执行去重

> **注意**：`context.txt` 约 19.3 GB，耗时最长，请耐心等待。

In [4]:
results = {}

for fname in FILES:
    src = RAW_DIR   / fname
    dst = DEDUP_DIR / fname

    if not src.exists():
        print(f"[跳过] {fname} 不存在，路径：{src}")
        continue

    if dst.exists():
        print(f"[跳过] {fname} 已存在于输出目录，如需重新生成请手动删除 {dst}")
        continue

    print(f"\n开始处理: {fname}  (源文件大小: {src.stat().st_size / 1e9:.2f} GB)")
    info = dedup_file(src, dst, KEEP_LINES)
    results[fname] = info
    print(f"完成: {fname}  写入 {info['lines_written']:,} 行  "
          f"耗时 {info['elapsed_s']:.1f}s  "
          f"输出大小: {dst.stat().st_size / 1e9:.2f} GB")

print("\n" + "=" * 60)
print("全部处理完成。")
total_time = sum(v['elapsed_s'] for v in results.values())
print(f"总耗时: {total_time/60:.1f} 分钟")


开始处理: meta.txt  (源文件大小: 0.40 GB)


meta.txt:   0%|          | 0.00/1.37M [00:00<?, ?行/s]

完成: meta.txt  写入 1,367,753 行  耗时 1.6s  输出大小: 0.20 GB

开始处理: add.txt  (源文件大小: 0.26 GB)


add.txt:   0%|          | 0.00/1.37M [00:00<?, ?行/s]

完成: add.txt  写入 1,367,753 行  耗时 1.5s  输出大小: 0.13 GB

开始处理: rem.txt  (源文件大小: 0.25 GB)


rem.txt:   0%|          | 0.00/1.37M [00:00<?, ?行/s]

完成: rem.txt  写入 1,367,753 行  耗时 1.5s  输出大小: 0.13 GB

开始处理: context.txt  (源文件大小: 20.69 GB)


context.txt:   0%|          | 0.00/1.37M [00:00<?, ?行/s]

完成: context.txt  写入 1,367,753 行  耗时 22.1s  输出大小: 10.35 GB

全部处理完成。
总耗时: 0.4 分钟


## Cell 5 — 结果验证

验证输出文件的行数正确，并抽样对比输入与输出内容一致。

In [5]:
def count_lines(path: Path) -> int:
    """流式统计文件行数，避免全量载入内存。"""
    n = 0
    with open(path, 'rb') as f:  # 二进制模式更快
        for chunk in iter(lambda: f.read(1 << 20), b''):
            n += chunk.count(b'\n')
    return n


def spot_check(src: Path, dst: Path, rows: list) -> bool:
    """抽样对比 src 与 dst 的指定行内容是否一致。"""
    targets = set(rows)
    src_lines, dst_lines = {}, {}

    with open(src, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, start=1):
            if i in targets:
                src_lines[i] = line
            if i > max(targets):
                break

    with open(dst, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, start=1):
            if i in targets:
                dst_lines[i] = line
            if i > max(targets):
                break

    all_ok = True
    for r in sorted(rows):
        match = src_lines.get(r) == dst_lines.get(r)
        if not match:
            all_ok = False
        print(f"  行 {r:>9,}: {'✓' if match else '✗ 不一致！'}")
    return all_ok


SPOT_ROWS = [1, 100, 10_000, 500_000, KEEP_LINES]

print("=" * 60)
print("验证输出文件行数 & 内容抽样")
print("=" * 60)

for fname in FILES:
    dst = DEDUP_DIR / fname
    src = RAW_DIR   / fname
    if not dst.exists():
        print(f"\n[跳过验证] {fname} 尚未生成")
        continue

    print(f"\n── {fname} ──")

    # 行数验证
    actual = count_lines(dst)
    line_ok = (actual == KEEP_LINES)
    print(f"  行数: {actual:,}  (期望 {KEEP_LINES:,})  {'✓' if line_ok else '✗ 行数错误！'}")

    # 内容抽样（仅对小文件或 meta 做详细抽样，context 只验证行数）
    if fname != "context.txt":
        print(f"  内容抽样（行号）:")
        spot_check(src, dst, SPOT_ROWS)
    else:
        print("  context.txt 文件过大，跳过内容抽样，仅验证行数。")

print("\n" + "=" * 60)
print("验证完成。")

验证输出文件行数 & 内容抽样

── meta.txt ──
  行数: 1,367,753  (期望 1,367,753)  ✓
  内容抽样（行号）:
  行         1: ✓
  行       100: ✓
  行    10,000: ✓
  行   500,000: ✓
  行 1,367,753: ✓

── add.txt ──
  行数: 1,367,753  (期望 1,367,753)  ✓
  内容抽样（行号）:
  行         1: ✓
  行       100: ✓
  行    10,000: ✓
  行   500,000: ✓
  行 1,367,753: ✓

── rem.txt ──
  行数: 1,367,753  (期望 1,367,753)  ✓
  内容抽样（行号）:
  行         1: ✓
  行       100: ✓
  行    10,000: ✓
  行   500,000: ✓
  行 1,367,753: ✓

── context.txt ──
  行数: 1,367,753  (期望 1,367,753)  ✓
  context.txt 文件过大，跳过内容抽样，仅验证行数。

验证完成。
